<a href="https://colab.research.google.com/github/hridayrjain/BIS-Lab/blob/main/1BM24CS119_%20Robotics%20and%20automation_%20Cuckoo%20Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Hriday R Jain

---

1BM24CS119

Application: Robotics and automation

Problem statement: An autonomous robot must travel from a starting position to a
target position in an environment containing obstacles. Develop a CS algorithm to
optimize the robot&#39;s path while avoiding collisions and minimizing travel distance.

Objective: Find a feasible collision-free path with minimum length or travel time.

In [2]:
import random

# 1. Grid setup
N = 8
START = (0, 0)
GOAL = (7, 7)

OBSTACLES = {
    (1, 3), (2, 3), (3, 3), (4, 3),
    (5, 3), (5, 4), (5, 5)
}

# 2. User inputs
NESTS = int(input("Enter number of nests: "))
ITERATIONS = int(input("Enter number of iterations: "))


# 3. Generate a random collision-free path
def generate_path(start, blocked):
    if start == GOAL:
        return [start]

    visited = blocked | {start}

    def dfs(pos):
        if pos == GOAL:
            return [pos]

        r, c = pos
        neighbors = [
            (r+1, c), (r-1, c),
            (r, c+1), (r, c-1)
        ]
        random.shuffle(neighbors)

        for nr, nc in neighbors:
            nxt = (nr, nc)

            if (0 <= nr < N and 0 <= nc < N
                and nxt not in OBSTACLES
                and nxt not in visited):

                visited.add(nxt)
                result = dfs(nxt)

                if result:
                    return [pos] + result

                visited.remove(nxt)

        return None

    return dfs(start)


# 4. Fitness evaluation
def fitness(path):
    return 1 / len(path) if path else 0


# 5. Lévy flight: modify an existing path
def levy_flight(path):
    index = random.randint(0, len(path)-2)

    # Random jump length inspired by Lévy flight
    step = min(
        int(1 / (random.random() ** (1/1.5))),
        len(path)-1-index
    )

    index += step

    prefix = path[:index+1]
    suffix = generate_path(prefix[-1], set(prefix[:-1]))

    if suffix:
        return prefix[:-1] + suffix

    return path


# 6. Initialize nests
nests = [generate_path(START, set()) for _ in range(NESTS)]
best_path = min(nests, key=len)

# 7. Cuckoo Search iterations
for i in range(ITERATIONS):

    # Generate new solution using Lévy flight
    new_path = levy_flight(random.choice(nests))

    # Replace a weaker nest if a better solution is found
    worst = max(range(NESTS), key=lambda j: len(nests[j]))

    if fitness(new_path) > fitness(nests[worst]):
        nests[worst] = new_path

    # Abandon 25% of poor nests
    nests.sort(key=len)

    for j in range(max(1, NESTS // 4)):
        nests[-(j+1)] = generate_path(START, set())

    # Update best solution
    current = min(nests, key=len)

    if len(current) < len(best_path):
        best_path = current

    if i == 0 or i == ITERATIONS-1:
        print("Iteration", i+1,
              "- Best path length:", len(best_path)-1)


# 8. Display result
print("\nBest Path:", best_path)
print("Path Length:", len(best_path)-1)

grid = [['.' for _ in range(N)] for _ in range(N)]

for r, c in OBSTACLES:
    grid[r][c] = '#'

for r, c in best_path:
    grid[r][c] = '*'

grid[0][0] = 'S'
grid[7][7] = 'G'

print("\nRobot Path:")
for row in grid:
    print(' '.join(row))

Enter number of nests: 20
Enter number of iterations: 50
Iteration 1 - Best path length: 14
Iteration 50 - Best path length: 14

Best Path: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 4), (2, 4), (3, 4), (3, 5), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7), (7, 7)]
Path Length: 14

Robot Path:
S * * * * . . .
. . . # * . . .
. . . # * . . .
. . . # * * . .
. . . # . * * .
. . . # # # * *
. . . . . . . *
. . . . . . . G
